# Compression / codec impact across vertex count

Sweep the `compressor=` kwarg across five codec configurations *and*
vertex count `N`, plotted as lines on log-log axes with 95% confidence
bands.  The library default is **no compression** (`BytesCodec` only)
for the fastest cloud-write path; this benchmark surfaces what each
compressed alternative buys you for what cost, as a function of scale.

To isolate codec scaling from chunk-size effects, **per-chunk byte
content is held roughly constant across `N`** by shrinking
`chunk_shape` as `N` grows — every chunk holds ~`TARGET_VERTS`
vertices in expectation.  So as `N` increases, the **number of
chunks** rises proportionally rather than each chunk getting bigger.
This means the per-chunk codec workload is the same at every `N` and
only the *aggregate* work (CPU, disk, file count) scales.

| Label | `compressor=` value |
|-------|--------------------|
| `none` (default) | `None` — uncompressed chunks, fast async PUT path |
| `zstd` | `'zstd'` — zarr v3's default compressor (Zstd l0) |
| `blosc_l1` | `[{...}]` Blosc + Zstd level 1, no shuffle |
| `blosc_l5` | `[{...}]` Blosc + Zstd level 5, no shuffle |
| `blosc_l5_bitshuffle` | `'blosc'` — Blosc + Zstd level 5 + BitShuffle |

Compression CPU usually pays off on disk; bitshuffle is expected to
beat plain Zstd on float32 positions.  See
[`docs/spec/foundations/codec_pipeline.md`](../docs/spec/foundations/codec_pipeline.md)
for the kwarg semantics.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points

SIZES        = [10_000, 100_000, 1_000_000]
TARGET_VERTS = 5_000             # ≈ 60 KB raw float32 per chunk
DOMAIN       = 1000.0
SEED         = 0


def _chunk_shape_for(n):
    """Pick a uniform 3D ``chunk_shape`` so each chunk holds about
    ``TARGET_VERTS`` vertices in expectation for uniform random
    positions on ``[0, DOMAIN)^3``.

    Returns ``(chunk_shape, bin_shape)``.  ``bin_shape`` is 1/4 of
    ``chunk_shape`` (matches the convention used by the other
    benchmarks).  Capped at the domain edge so we never overshoot.
    """
    side = DOMAIN * (TARGET_VERTS / n) ** (1 / 3)
    side = min(side, DOMAIN)
    return (side,) * 3, (side / 4.0,) * 3


CONFIGS = [
    ('none',                None),
    ('zstd',                'zstd'),
    ('blosc_l1',            [{'name': 'blosc', 'configuration': {
                                'cname': 'zstd', 'clevel': 1,
                                'shuffle': 'noshuffle',
                                'typesize': 4, 'blocksize': 0}}]),
    ('blosc_l5',            [{'name': 'blosc', 'configuration': {
                                'cname': 'zstd', 'clevel': 5,
                                'shuffle': 'noshuffle',
                                'typesize': 4, 'blocksize': 0}}]),
    ('blosc_l5_bitshuffle', 'blosc'),
]

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)

# raw[metric][config_label] is shape (len(SIZES), N_RUNS).
metrics = ['write_s', 'read_all_s']
raw = {
    m: {label: np.zeros((len(SIZES), N_RUNS)) for label, _ in CONFIGS}
    for m in metrics
}
disk_MB = {label: np.zeros(len(SIZES)) for label, _ in CONFIGS}
per_size_chunk_shape = []

for i, n in enumerate(SIZES):
    chunk_shape, bin_shape = _chunk_shape_for(n)
    per_size_chunk_shape.append(chunk_shape[0])
    print(f'N = {n:>9,}  chunk_shape = {chunk_shape[0]:6.1f}  '
          f'≈ {n / max(1, (DOMAIN / chunk_shape[0]) ** 3):6.0f} verts/chunk')
    positions = rng.uniform(0, DOMAIN, (n, 3)).astype(np.float32)
    for label, compressor in CONFIGS:
        for run in range(N_RUNS):
            store = _new_store(f'codec_{label}_n{n}_run{run}')
            t_w, _ = _time(
                write_points, store, positions,
                chunk_shape=chunk_shape, bin_shape=bin_shape,
                compressor=compressor,
            )
            t_r, _ = _time(read_points, store)
            raw['write_s'][label][i, run]    = t_w
            raw['read_all_s'][label][i, run] = t_r
            if run == 0:
                disk_MB[label][i] = _store_bytes(store) / 1e6
            shutil.rmtree(Path(store).parent, ignore_errors=True)

# Tidy long-form df: one row per (N, config) holding fold change vs the
# ``none`` (uncompressed) baseline at the same N.  Time-metric folds are
# computed per-run paired (same run index for both numerator and
# denominator), then summarised as mean + 95% CI; disk fold is a single
# deterministic measurement per (N, config).
rows = []
for i, n in enumerate(SIZES):
    for label, _ in CONFIGS:
        row = {
            'N': n,
            'chunk_shape': round(per_size_chunk_shape[i], 1),
            'config': label,
        }
        for m in metrics:
            # Pairwise ratio across runs: shape (N_RUNS,).
            ratio = raw[m][label][i] / raw[m]['none'][i]
            mean, hw = _mean_ci95(ratio)
            row[f'{m}_fold_mean'] = round(mean, 4)
            row[f'{m}_fold_hw']   = round(hw,   4)
        row['disk_fold'] = round(disk_MB[label][i] / disk_MB['none'][i], 4)
        rows.append(row)
df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

PANELS = [
    ('Write time fold change',  'write_s_fold'),
    ('Read time fold change',   'read_all_s_fold'),
]
COLORS = {
    'none':                '0.5',          # gray reference line at 1.0
    'zstd':                'C1',
    'blosc_l1':            'C2',
    'blosc_l5':            'C3',
    'blosc_l5_bitshuffle': 'C4',
}

def _plot_fold(ax, key_prefix, title, with_ci):
    """Plot fold-change lines per config on log-log axes.

    ``key_prefix`` is ``'write_s_fold'``, ``'read_all_s_fold'``, or
    ``'disk'``.  When ``with_ci``, draws a shaded 95% CI band around
    each mean line (only valid for time metrics that are sampled
    per-run).
    """
    for label, _ in CONFIGS:
        sub = df[df['config'] == label].sort_values('N')
        if label == 'none':
            # Self-ratio is exactly 1 by construction; draw a faint
            # horizontal reference at y=1 instead of a redundant data
            # series.
            continue
        if with_ci:
            mean = sub[f'{key_prefix}_mean'].values
            hw   = sub[f'{key_prefix}_hw'].values
            ax.fill_between(
                sub['N'], mean - hw, mean + hw,
                color=COLORS[label], alpha=0.2,
            )
        else:
            mean = sub[key_prefix].values
        ax.loglog(
            sub['N'], mean, 'o-',
            color=COLORS[label], label=label,
        )
    ax.axhline(1.0, color=COLORS['none'], linestyle='--',
               linewidth=1, label='none (1.0×)')
    ax.set_xlabel('N (vertices)')
    ax.set_ylabel('fold change vs none')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)

# Time panels — fold change with per-run paired CI.
for ax, (title, key) in zip(axes[:2], PANELS):
    _plot_fold(ax, key, title, with_ci=True)

# Disk panel — deterministic single measurement; no CI.
_plot_fold(axes[2], 'disk_fold', 'Disk size fold change', with_ci=False)

axes[0].legend(loc='best', fontsize=8)
fig.suptitle(
    f'Compression impact — fold change vs uncompressed default '
    f'({N_RUNS} runs, 95% CI on time metrics)',
)
plt.tight_layout()